# 05 · The AnnData object

Stage 1 built seven files and this stage opens one of them.

| | |
|---|---|
| **1** | Open it, and print the slots |
| **2** | The eight slots, and what each is for |
| **3** | Indexing: names, masks, and the view |
| **4** | The `obs` column type that changes your answers |
| **5** | Which file to open for which question |

In [5]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

plt.rcParams.update({          # the house style, no package needed
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
    "legend.frameon": False, "pdf.fonttype": 42, "ps.fonttype": 42,
})
pd.set_option("display.width", 140)

# The one path to set. Point MCS2026_DATA at the folder holding the tables, or edit this.
DATA = Path(os.environ.get("MCS2026_DATA", "/cluster/work/liberali/COURSE/mcs2026/tables"))


## 1 · Open it, and print the slots

The first cell of every notebook you write, before you touch anything. Someone else made
this file — possibly you, six weeks ago — and this is how you find out what they did.

In [6]:
cells = sc.read_h5ad(DATA / "mcs2026_controls.h5ad")
cells

AnnData object with n_obs × n_vars = 97177 × 38
    obs: 'label', 'well_name', 'ROI', 'is_border_internal', 'is_border_external', 'Organoid_ID', 'timepoint_h', 'condition', 'condition_code', 'well', 'row', 'column', 'is_outlier_condition', 'area', 'eccentricity', 'solidity', 'extent', 'roundness', 'x_in_well', 'y_in_well', 'dapi'
    var: 'family', 'statistic', 'channel', 'round', 'marker', 'species', 'intensity_threshold', 'haralick_threshold', 'failed', 'is_reference', 'is_structural', 'theme', 'column'
    uns: 'decode_report', 'dropped_wells', 'panels', 'provenance', 'subset_conditions', 'subset_of', 'subset_reason', 'themes'
    layers: 'raw'

## 2 · The eight slots, and what each is for

```{image} ../../images/clean_object_light.svg
:class: only-light
:alt: The analysis AnnData: X holding normalised values, a raw layer, an annotated var table, a tidy obs table, and provenance in uns.
```
```{image} ../../images/clean_object_dark.svg
:class: only-dark
:alt: The analysis AnnData: X holding normalised values, a raw layer, an annotated var table, a tidy obs table, and provenance in uns.
```

| slot | shape | holds |
|---|---|---|
| `X` | `n_obs × n_vars` | the measurements — one number per cell per marker |
| `obs` | `n_obs` rows | what you know about each **cell** |
| `var` | `n_vars` rows | what you know about each **marker** |
| `layers` | `n_obs × n_vars` | alternative versions of `X` — same shape, different numbers |
| `obsm` | `n_obs × anything` | per-cell matrices: embeddings, PCA coordinates |
| `varm` | `n_vars × anything` | per-**marker** matrices: PCA loadings |
| `obsp` | `n_obs × n_obs` | per-cell-pair matrices: neighbour graphs, sparse |
| `uns` | anything | everything else — parameters, colours, provenance |

**`X` and `layers` are the same cells and the same markers.** Here `X` is normalised and
`layers["raw"]` is what the microscope measured, so you can always get back:

In [9]:
print(f"  X          {cells.X.shape}  {cells.X.dtype}  "
      f"mean {cells.X.mean():+.3f}   range {cells.X.min():+.1f} to {cells.X.max():+.1f}")
raw = cells.layers["raw"]
print(f"  layers.raw {raw.shape}  {raw.dtype}  "
      f"mean {raw.mean():8.1f}   range {raw.min():.0f} to {raw.max():.0f}")

  X          (97177, 38)  float32  mean +0.097   range -10.2 to +14.1
  layers.raw (97177, 38)  float32  mean    205.1   range 0 to 39569


**`obs` and `var` are ordinary pandas DataFrames** whose row order is locked to the
matrix. Anything you can do to a DataFrame you can do to them.

In [10]:
cells.obs.head(3)

,label,well_name,ROI,is_border_internal,is_border_external,Organoid_ID,timepoint_h,condition,condition_code,well,...,column,is_outlier_condition,area,eccentricity,solidity,extent,roundness,x_in_well,y_in_well,dapi
Organoid_ID,,,,,,,,,,,,,,,,,,,,,
HNES1-MP-PL-1-B02-16,17,B02,FOV_1,False,False,HNES1-MP-PL-1-B02-16,36,PBS,Cnd17,B02,...,2,False,2303.0,0.894302,0.954414,0.461338,0.918061,39.322189,1714.547119,444.915314
HNES1-MP-PL-1-B02-17,18,B02,FOV_1,False,False,HNES1-MP-PL-1-B02-17,36,PBS,Cnd17,B02,...,2,False,2542.0,0.938046,0.775473,0.378555,1.290050,65.345787,1620.319824,88.717941
HNES1-MP-PL-1-B02-18,19,B02,FOV_1,False,False,HNES1-MP-PL-1-B02-18,36,PBS,Cnd17,B02,...,2,False,4765.0,0.817354,0.900586,0.548205,1.146383,67.862541,1747.877441,369.934509


In [11]:
cells.var.head(3)

,family,statistic,channel,round,marker,species,intensity_threshold,haralick_threshold,failed,is_reference,is_structural,theme,column
beta-Catenin,Intensity,mean_intensity,FITC,0,beta-Catenin,Mouse,1000.0,3000.0,False,False,False,signaling,cells_Intensity_mean_intensity_FITC_0
Foxo3a,Intensity,mean_intensity,Texas Red,0,Foxo3a,Rabbit,500.0,750.0,False,False,False,signaling,cells_Intensity_mean_intensity_Texas Red_0
FGFR1,Intensity,mean_intensity,Cy5,1,FGFR1,Rabbit,500.0,2000.0,False,False,False,signaling,cells_Intensity_mean_intensity_Cy5_1


**`obsm` holds matrices that share the cell axis but not the marker axis.** A PCA has one
row per cell and 20 columns that are not markers, so it cannot go in `X` and does not
belong in `obs`. **`obsp` holds `n_obs × n_obs` matrices** — a neighbour graph is one
number per *pair* of cells.

Chapters 06 to 09 are what fill them, one slot each. Whether they are filled *now* depends
on how far through the course this file has been.

## 3 · Indexing: names, masks, and the view

`adata[cells, markers]`, and each half accepts names, positions or a boolean mask.

In [14]:
print(f"  by marker name        {cells[:, 'Oct4'].shape}")
print(f"  by several            {cells[:, ['Oct4', 'Nanog', 'Sox2']].shape}")
print(f"  by cell mask          {cells[cells.obs.condition == 'DMSO'].shape}")
print(f"  by both               {cells[cells.obs.timepoint_h == 84, 'GATA4'].shape}")
print(f"  by var lookup         {cells[:, cells.var.theme == 'signaling'].shape}")

  by marker name        (97177, 1)
  by several            (97177, 3)
  by cell mask          (62885, 38)
  by both               (43788, 1)
  by var lookup         (97177, 9)


Every one of those is a **view** — a window onto the original, costing no memory until
you materialise it.

In [15]:
view = cells[cells.obs.condition == "DMSO"]
print(f"  view  is_view={view.is_view}   shares memory with the original")
copy = view.copy()
print(f"  copy  is_view={copy.is_view}   {copy.n_obs * copy.n_vars * 4 / 1e6:.1f} MB of its own")

  view  is_view=True   shares memory with the original
  copy  is_view=False   9.6 MB of its own


:::{warning}
**Writing to a view does not do what you expect.** Assigning into `view.obs` raises, or
silently writes to a copy that is then discarded, depending on the version and the slot.

The rule that always works: **call `.copy()` the moment you intend to modify something.**
Read from views freely; never write to one.
:::

In [16]:
subset = cells[cells.obs.timepoint_h.astype(int) >= 60].copy()

subset.obs["late"] = True

print(f"  {subset.n_obs:,} cells, and now an extra obs column: {'late' in subset.obs}")

  71,229 cells, and now an extra obs column: True


## 4 · The `obs` column type that changes your answers

Half of `obs` is **categorical**, not string. That is not cosmetic.

In [17]:
print(cells.obs.dtypes.astype(str).to_string())

label                      int64
well_name               category
ROI                     category
is_border_internal          bool
is_border_external          bool
Organoid_ID               object
timepoint_h             category
condition               category
condition_code          category
well                    category
row                     category
column                     int64
is_outlier_condition        bool
area                     float32
eccentricity             float32
solidity                 float32
extent                   float32
roundness                float32
x_in_well                float32
y_in_well                float32
dapi                     float32


A categorical remembers every category it was declared with, whether or not any rows are
left holding it. Subsetting an `AnnData` tidies that up for you. Subsetting `obs` as a
plain DataFrame — which you will do constantly, because it *is* a DataFrame — does not.

Same filter, two ways:

In [18]:
# Keep one of the two vehicles. PBS then has no rows left -- which is the point.
mask = cells.obs.condition == "DMSO"
print(f"  AnnData subset   : {len(cells[mask].obs.condition.cat.categories)} categories")
print(f"  DataFrame subset : {len(cells.obs[mask].condition.cat.categories)} categories")

  AnnData subset   : 1 categories
  DataFrame subset : 2 categories


And that difference decides what `groupby` gives back:

In [19]:
frame = cells.obs[mask].assign(Oct4=np.asarray(cells[mask, "Oct4"].X).ravel())
pd.DataFrame({
    "cells (observed=False)": frame.groupby("condition", observed=False).size(),
    "mean Oct4 (observed=False)": frame.groupby("condition", observed=False).Oct4.mean().round(3),
}).head(8)

,cells (observed=False),mean Oct4 (observed=False)
condition,,
DMSO,62885,0.0
PBS,0,NaN


In [20]:
frame.groupby("condition", observed=True).agg(cells=("Oct4", "size"),
                                              mean_Oct4=("Oct4", "mean")).round(3)

,cells,mean_Oct4
condition,,
DMSO,62885,0.0


:::{warning}
**Sixteen of those rows are conditions that are not in the subset**, reported as 0 cells
and a `NaN` mean. Put the first table in a bar chart and you get two real bars and sixteen
at zero — which reads as "these compounds did nothing", the exact opposite of "these
compounds are not in this data".

**Pass `observed=True` to every `groupby` on a categorical.** Recent pandas warns and will
eventually change the default; until then, the failure is silent and looks like a result.

`timepoint_h` is also an **ordered** categorical, which is right for plotting and wrong for
arithmetic — pandas will not subtract categories. Cast it: `obs.timepoint_h.astype(int)`.
:::

## 5 · Which file to open for which question

Choosing the wrong one is the most common way to get a confidently wrong answer in Part 3.
These are the ones you open; `mcs2026_slim.h5ad` and `mcs2026_qc.h5ad` are Stage 1's own
intermediates and `mcs2026_full.h5ad` is the archive you go back to for texture.

In [21]:
paths = {
    "controls  (this stage)":       DATA / "mcs2026_controls.h5ad",
    "intensity (all cells)":        DATA / "mcs2026_intensity.h5ad",
    "sketch    (all 18, reduced)":  DATA / "mcs2026_sketch.h5ad",
    "full      (wide, 2,587 cols)": DATA / "mcs2026_full.h5ad",
}
pd.DataFrame([
    {"file": path.name, "MB on disk": round(path.stat().st_size / 1e6, 1)}
    for path in paths.values()
], index=list(paths)).rename_axis("open it for")

,file,MB on disk
open it for,,
controls (this stage),mcs2026_controls.h5ad,37.4
intensity (all cells),mcs2026_intensity.h5ad,248.2
"sketch (all 18, reduced)",mcs2026_sketch.h5ad,11.8
"full (wide, 2,587 cols)",mcs2026_full.h5ad,5983.8


| file | one row per | open it when |
|---|---|---|
| `mcs2026_controls.h5ad` | cell | **you are in Stage 2.** DMSO and PBS, every cell of them |
| `mcs2026_intensity.h5ad` | cell | you need all 18 conditions — counting, proportions, projecting labels |
| `mcs2026_sketch.h5ad` | cell | you are embedding all 18 conditions and 653,000 cells will not fit |
| `mcs2026_full.h5ad` | cell | you need **texture**, or a marker statistic other than the mean |

:::{important}
**Stage 2 opens the controls and nothing else.** Every method in the chapters that follow
is learned on the two conditions that received no compound — so if a technique appears to
find something, the finding is a property of the technique.

It is also the one subset small enough to use whole. 32 wells fit in memory and in
patience; the 653,000-cell table does not, which is why Part 4 has a sketch and this stage
does not need one.
:::

## The habit

One cell, at the top of every notebook, before anything else:

```python
cells = sc.read_h5ad(path)
print(cells)                       # the slots
print(cells.uns["provenance"])     # what was done to it
print(cells.obs.dtypes)            # what will need casting
```


---

## Summary

| slot | holds |
|---|---|
| `X` / `layers` | the same cells and the same markers, different numbers |
| `obs` / `var` | ordinary DataFrames, locked to the row and column axes |
| `obsm` / `obsp` | per-cell matrices, and per-cell-**pair** matrices |
| `varm` | per-marker matrices — PCA loadings live here, not in `var` |
| `uns` | everything that fits nowhere else, including `provenance` |

Three habits that prevent most of the confusing errors:

- **Read from views freely; call `.copy()` the moment you intend to write.**
- **Pass `observed=True` to every `groupby` on a categorical**, or conditions that are not
  in the subset come back as zeros that read like a result.
- **Cast `timepoint_h` with `.astype(int)` before any arithmetic** — it is an *ordered*
  categorical, which is right for plotting and wrong for subtraction.




---

**Next:** [06 · PCA](06_pca.ipynb).